In [ ]:
# 공통 경로와 패키지
from pathlib import Path
import os
import sys

project_root = next(
    (folder for folder in (Path.cwd(), *Path.cwd().parents)
     if (folder / "project_paths.py").is_file()),
    None,
)
if project_root is None:
    raise RuntimeError("프로젝트 폴더에서 노트북을 실행하세요.")
if str(project_root) in sys.path:
    sys.path.remove(str(project_root))
sys.path.insert(0, str(project_root))
sys.modules.pop("project_paths", None)
from project_paths import DATA_ROOT, RAW_DIR, PROCESSED_DIR, OUTPUT_DIR, PATIENT_MAPPING_PATH, deidentified_path

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import os
import glob
import pandas as pd
from datetime import timedelta

table_dir = str(PROCESSED_DIR / 'AEGIS-Basic-Info/de')

# 환자별 노이즈 정보
patient_mapping_df = pd.read_csv(str(PATIENT_MAPPING_PATH), sep=",", quotechar ='"', lineterminator = '\n',dtype = {'pt_id': str})

# 노이즈 매핑 딕셔너리
patient_mapping_df['new_pt_id'] = patient_mapping_df['new_pt_id'].astype(str)

patient_id_map = patient_mapping_df.set_index('pt_id')['new_pt_id'].to_dict()
date_offset_map = patient_mapping_df.set_index('new_pt_id')['date_noise'].astype(int).to_dict()
time_offset_map = patient_mapping_df.set_index('new_pt_id')['time_noise'].astype(int).to_dict()

csv_files = glob.glob(os.path.join(table_dir, '**/*.csv'), recursive = True)
print(f"[INFO] 총 처리 대상 파일 수 : {len(csv_files)}")


In [ ]:
# pt_id 치환
for file in csv_files:
    try:
        basic_info_df = pd.read_csv(file, dtype={'pt_id':str},sep=',', quotechar='"', lineterminator='\n')

        if 'pt_id' in basic_info_df.columns:
            basic_info_df['pt_id'] = basic_info_df['pt_id'].map(patient_id_map).fillna(basic_info_df['pt_id'])

        basic_info_df.to_csv(file, index=False, encoding = 'utf-8')
    except Exception as e:
        print(f"[ERROR] pt_id 오류: {file} → {e}")


In [ ]:
# hos_dt ±7일 ±30분
for file in csv_files:
    try:
        basic_info_df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')

        if 'hos_dt' in basic_info_df.columns:
            basic_info_df['hos_dt'] = pd.to_datetime(basic_info_df['hos_dt'], errors='coerce')
            basic_info_df['hos_dt'] = basic_info_df['hos_dt'] + basic_info_df['pt_id'].map(lambda x: timedelta(days=date_offset_map.get(x, 0), minutes=time_offset_map.get(x, 0)))

            basic_info_df.to_csv(file, index=False, encoding = 'utf-8')
    except Exception as e:
        print(f"[ERROR] hos_dt 오류: {file} → {e}")


In [ ]:
# birth_date → day=1 고정
for file in csv_files:
    try:
        basic_info_df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')
        if 'birth_date' in basic_info_df.columns:
            basic_info_df['birth_date'] = pd.to_datetime(basic_info_df['birth_date'], errors='coerce')
            basic_info_df['birth_date'] = basic_info_df['birth_date'].apply(lambda x: x.replace(day=1) if pd.notnull(x) else pd.NaT)

            basic_info_df.to_csv(file, index=False, encoding = 'utf-8')
    except Exception as e:
        print(f"[ERROR] birth_date 오류: {file} → {e}")


In [ ]:
# death_date ±7일
for file in csv_files:
    try:
        basic_info_df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')
        if 'death_date' in basic_info_df.columns:
            basic_info_df['death_date'] = pd.to_datetime(basic_info_df['death_date'], errors='coerce')
            basic_info_df['death_date'] = basic_info_df['death_date'] + basic_info_df['pt_id'].map(lambda x: timedelta(days=date_offset_map.get(x, 0)))

            basic_info_df.to_csv(file, index=False, encoding = 'utf-8')
    except Exception as e:
        print(f"[ERROR] death_date 오류: {file} → {e}")


In [ ]:
# death_est_date ±7일
for file in csv_files:
    try:
        basic_info_df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')
        if 'death_est_date' in basic_info_df.columns:
            basic_info_df['death_est_date'] = pd.to_datetime(basic_info_df['death_est_date'], errors='coerce')
            basic_info_df['death_est_date'] = basic_info_df['death_est_date'] + basic_info_df['pt_id'].map(lambda x: timedelta(days=date_offset_map.get(x, 0)))

            basic_info_df.to_csv(file, index=False, encoding = 'utf-8')
    except Exception as e:
        print(f"[ERROR] death_est_date 오류: {file} → {e}")


In [ ]:
# Region_cd 범주화
for file in csv_files:
    try:
        basic_info_df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')
        if 'Region_cd' in basic_info_df.columns:
            basic_info_df['Region_cd'] = basic_info_df['Region_cd'].apply(
                lambda x: x if pd.isna(x) or x == 'KR' else 'Others')

            basic_info_df.to_csv(file, index=False, encoding = 'utf-8')
    except Exception as e:
        print(f"[ERROR] Region_cd 오류: {file} → {e}")


In [ ]:
# Region_nm 범주화
for file in csv_files:
    try:
        basic_info_df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')
        if 'Region_nm' in basic_info_df.columns:
            basic_info_df['Region_nm'] = basic_info_df['Region_nm'].apply(
                lambda x: x if pd.isna(x) or x == '대한민국' else '국외')

            basic_info_df.to_csv(file, index=False, encoding = 'utf-8')
    except Exception as e:
        print(f"[ERROR] Region_nm 오류: {file} → {e}")


In [ ]:
# 민감 정보 컬럼 삭제
columns_to_drop = ['abroad', 'zip_code', 'privacy', 'sensitive', 'unique']
for file in csv_files:
    try:
        basic_info_df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')
        basic_info_df.drop(columns=[col for col in columns_to_drop if col in basic_info_df.columns], inplace=True)

        basic_info_df.to_csv(file, index=False, encoding = 'utf-8')
    except Exception as e:
        print(f"[ERROR] 컬럼 삭제 오류: {file} → {e}")


In [ ]:
# 특이 정보 추출 (death_cause_cd, death_caluse_nm N수 5 이하)

rare_combinations_all = []
for file in csv_files:
    try:
        basic_info_df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')

        if {'death_cause_cd', 'death_cause_nm'}.issubset(basic_info_df.columns):
            combination_counts_df = basic_info_df.groupby(['death_cause_cd', 'death_cause_nm']).size().reset_index(name = 'count')
            rare_combinations = combination_counts_df[combination_counts_df['count'] <= 5].copy()
            rare_combinations['source_file'] = os.path.basename(file)
            rare_combinations_all.append(rare_combinations)
        else:
            print(f"[WARN] 해당 컬럼 없음: {file}")

    except Exception as e:
        print(f"[ERROR] 오류 발생: {file} -> {e}")

if rare_combinations_all:
    rare_combinations_df = pd.concat(rare_combinations_all, ignore_index = True)
    rare_combinations_df = rare_combinations_df.sort_values(by='count', ascending = True).reset_index(drop = True)
else:
    print("[WARN] 조건에 맞는 조합 없음")
print(f"[INFO] 총 count 수: {rare_combinations_df['count'].sum()}")


In [ ]:
# 특이 정보 제거(사망원인 N수 5이하 제거)

import os
import glob
import pandas as pd

# 희귀 조합 (rare_combinations_df 기준)
rare_combinations = rare_combinations_df[['death_cause_cd', 'death_cause_nm']].dropna().drop_duplicates()

for file in csv_files:
    try:
        basic_info_df = pd.read_csv(file, dtype={'pt_id': str}, sep=',', quotechar='"', lineterminator='\n')

        if {'death_cause_cd', 'death_cause_nm'}.issubset(basic_info_df.columns):
            before_rows = len(basic_info_df)

            joined_table_df = basic_info_df.merge(rare_combinations, on = ['death_cause_cd', 'death_cause_nm'], how = 'left', indicator = True)
            filtered_table_df = joined_table_df[joined_table_df['_merge'] == 'left_only'].drop(columns = ['_merge'])

            after_rows = len(filtered_table_df)
            removed_count = before_rows - after_rows

            filtered_table_df.to_csv(file,index = False, encoding = 'utf-8')

            print(f"[INFO] {file}: {removed_count}개 행 제거됨 (남은 행: {after_rows})")

        else:
            print(f"[WARN] 해당 컬럼 없음: {file}")

    except Exception as e:
        print(f"[ERROR] 오류 발생: {file} → {e}")
